# EVE 310 - Lab 09: Batch processing — many buildings

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-multiple-files.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. List CSV files in a folder
2. Reuse the single-file analysis inside a for loop

## Before you start

- Run `uv sync` in the repository root, then launch this notebook with `uv run jupyter lab` (see `docs/setup.md`).
- Work through the cells in order. Cells marked **Your turn** are for you to complete.


The analysis in the single-file notebook is unchanged. The new idea is to discover every `water_*.csv` and repeat the plot for each building.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from eve310 import set_plot_style

set_plot_style()

from pathlib import Path


## 1. List CSV files


In [ ]:
data_dir = Path('../data')
csv_files = sorted(p for p in data_dir.iterdir() if p.suffix.lower() == '.csv')
csv_files


## 2. Worked example — loop


In [ ]:
months = ['January', 'February', 'March', 'April', 'May', 'June',
          'July', 'August', 'September', 'October', 'November', 'December']
col = 'Water ( Gallons )'

for file_path in csv_files:
    water_df = pd.read_csv(file_path)
    building = file_path.stem.replace('water_', '')
    water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
    water_df['Month'] = water_df['DateTime'].dt.month
    w_std = np.std(water_df[col], ddof=1)
    w_mean = np.mean(water_df[col])
    upper, lower = w_mean + 3 * w_std, w_mean - 3 * w_std
    water_df = water_df.loc[(water_df[col] < upper) & (water_df[col] > lower)]
    water_df.boxplot(column=col, by='Month', grid=False, rot=45)
    plt.xticks(range(1, 13), months)
    plt.ylabel('Consumption (gallons)')
    plt.xlabel('Month')
    plt.title(f'{building} water consumption by month')
    plt.suptitle('')
    plt.savefig(f'../figures/{building}_water_boxplot.png', bbox_inches='tight', dpi=200)
    plt.close()
    print('saved', building)


## 3. Wrap-up

Check `../figures/` for one box plot per building.
